# Cinemática directa y control del robot SCARA RRP (robot real)

En este Notebook se implementa la cinemática directa de un robot SCARA RRP con dos articulaciones rotacionales (q1, q2) y una prismática (d3), utilizando las dimensiones del prototipo físico.

El objetivo es:
- Calcular la posición cartesiana del efector final \((x, y, z)\) a partir de las variables articulares \((q1, q2, d3)\).
- Enviar esos mismos valores articulares a un Arduino mediante el puerto serial.
- Comprobar experimentalmente que el robot real se mueve de acuerdo con lo calculado y generar un GIF de evidencia.


## 1. Instalación y carga de librerías

En esta sección se prepara el entorno de trabajo:

- Se instala la librería **`pyserial`**, necesaria para que Python pueda comunicarse con el Arduino por el puerto serial.
- Se importan las librerías básicas:
  - `numpy` para operaciones matemáticas y trigonométricas.
  - `serial` (pyserial) para la comunicación con el Arduino.
  - `time` para agregar pausas y asegurar que el microcontrolador tenga tiempo de reiniciarse y moverse.


In [3]:
!pip install pyserial


In [13]:
import numpy as np
import serial
import time


## 2. Comunicación serial con Arduino

En esta parte se configura el puerto serial que conecta la computadora con el Arduino:

- **`PORT`** se ajusta al puerto COM donde está conectado el Arduino (por ejemplo, `COM5` en Windows).
- **`BAUD`** se fija en `115200`, que debe coincidir con el valor de `Serial.begin(115200);` en el programa de Arduino.
- Se crea el objeto `ser = serial.Serial(...)`, que abre el puerto y permite enviar y recibir datos.
- Se agrega un `time.sleep(2)` para darle tiempo al Arduino de reiniciarse cuando se abre el puerto.

El mensaje que aparece en pantalla (`Conectado a COMx a 115200 baudios`) confirma que la conexión se realizó correctamente.


In [5]:
# Ajusta el puerto a tu caso:
# - En Windows: "COM3", "COM4", etc.
# - En Linux: "/dev/ttyUSB0" o "/dev/ttyACM0"
PORT = "COM5"
BAUD = 115200

ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)  # pequeña pausa para que se reinicie el Arduino/ESP32

print(f"Conectado a {PORT} a {BAUD} baudios")


Conectado a COM5 a 115200 baudios


## 3. Modelo geométrico del robot SCARA RRP

Aquí se definen los parámetros geométricos del robot real y su **cinemática directa**:

- `L1` y `L2` son las longitudes de los eslabones horizontales del brazo (80 mm cada uno).
- `H_MIN` es la altura mínima del efector final respecto a la base (23.45 mm).

La función `fk_scara(q1_deg, q2_deg, d3_mm)` implementa las ecuaciones de cinemática directa del robot:

\[
x = L_1 \cos(q_1) + L_2 \cos(q_1 + q_2)
\]
\[
y = L_1 \sin(q_1) + L_2 \sin(q_1 + q_2)
\]
\[
z = H_{\min} + d_3
\]

donde:
- `q1_deg` y `q2_deg` se ingresan en **grados** y se convierten a radianes dentro de la función.
- `d3_mm` es el desplazamiento vertical (en mm) desde la altura mínima del efector.

La salida de la función son las coordenadas cartesianas \((x, y, z)\) en milímetros.


In [9]:
# Parámetros geométricos del SCARA (mm)
L1 = 80.0
L2 = 80.0
H_MIN = 23.45  # altura mínima del efector

def fk_scara(q1_deg, q2_deg, d3_mm):
    """
    Cinemática directa de un SCARA RRP.
    Entradas:
        q1_deg, q2_deg: ángulos en grados
        d3_mm: recorrido vertical desde la altura mínima, en mm
    Salidas:
        (x, y, z) en mm
    """
    q1 = np.deg2rad(q1_deg)
    q2 = np.deg2rad(q2_deg)

    x = L1 * np.cos(q1) + L2 * np.cos(q1 + q2)
    y = L1 * np.sin(q1) + L2 * np.sin(q1 + q2)
    z = H_MIN + d3_mm

    return x, y, z


## 4. Ingreso manual de las variables articulares

En esta sección el usuario introduce manualmente los valores de:

- `q1_deg`: ángulo de la articulación 1 (base) en grados.
- `q2_deg`: ángulo de la articulación 2 (codo) en grados.
- `d3_mm`: desplazamiento vertical del efector (articulación prismática) en milímetros.

Los valores se capturan mediante `input()` para poder realizar diferentes pruebas.  
Al final se muestran en pantalla los valores ingresados, lo cual sirve como verificación antes de hacer los cálculos y enviar los datos al Arduino.


In [49]:
q1_deg = float(input("Ingresa q1 [grados]: "))
q2_deg = float(input("Ingresa q2 [grados]: "))
d3_mm  = float(input("Ingresa d3 [mm de recorrido desde la posición más baja]: "))

print(f"Valores ingresados -> q1 = {q1_deg}°, q2 = {q2_deg}°, d3 = {d3_mm} mm")


Ingresa q1 [grados]:  -90
Ingresa q2 [grados]:  -90
Ingresa d3 [mm de recorrido desde la posición más baja]:  0


Valores ingresados -> q1 = -90.0°, q2 = -90.0°, d3 = 0.0 mm


## 5. Cálculo de la posición cartesiana del efector

Con los valores articulares ingresados, se llama a la función `fk_scara`:

```python
x_mm, y_mm, z_mm = fk_scara(q1_deg, q2_deg, d3_mm)


In [51]:
x_mm, y_mm, z_mm = fk_scara(q1_deg, q2_deg, d3_mm)

print(f"Posición final del efector (mm):")
print(f"x = {x_mm:.2f} mm")
print(f"y = {y_mm:.2f} mm")
print(f"z = {z_mm:.2f} mm")


Posición final del efector (mm):
x = -80.00 mm
y = -80.00 mm
z = 23.45 mm


In [41]:
# Armar mensaje en el formato acordado: q1;q2;d3\n
msg = f"{q1_deg:.2f};{q2_deg:.2f};{d3_mm:.2f}\n"
print("Enviando al Arduino:", msg.strip())

ser.write(msg.encode('ascii'))
time.sleep(0.1)

# Leer posible respuesta (eco) del Arduino
respuesta = ser.readline().decode(errors="ignore").strip()
print("Arduino dice:", respuesta)


Enviando al Arduino: 0.00;0.00;0.00
Arduino dice: Recibido q1=-90.00 q2=-90.00 d3=0.00 -> servos: 0, 0, 0


## 8. Generación de GIF de evidencia

Para documentar el comportamiento del sistema se genera un **GIF animado**:

- Se define una lista `trayectoria` con varias configuraciones \((q1, q2, d3)\) por las que pasa el robot.
- Para cada configuración:
  - Se calcula la cinemática directa \((x, y, z)\).
  - Se dibuja el robot SCARA en el plano XY utilizando `matplotlib`.
  - Se guarda la imagen en la carpeta `media/`.

Finalmente, con la librería `imageio` se combinan todas las imágenes en un archivo:

```python
gif_path = "media/scara_robot_fisico.gif"
imageio.mimsave(gif_path, frames, duration=0.12)


In [1]:
!pip install imageio matplotlib


In [33]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import os

# --- Aseguramos parámetros y FK (aunque ya los tengas arriba, no pasa nada) ---
L1 = 80.0
L2 = 80.0
H_MIN = 23.45

def fk_scara(q1_deg, q2_deg, d3_mm):
    q1 = np.deg2rad(q1_deg)
    q2 = np.deg2rad(q2_deg)
    x = L1 * np.cos(q1) + L2 * np.cos(q1 + q2)
    y = L1 * np.sin(q1) + L2 * np.sin(q1 + q2)
    z = H_MIN + d3_mm
    return x, y, z

# --- Posiciones clave (puedes cambiar estos 3 puntos a los de tus pruebas) ---
trayectoria = [
    (90.0,   90.0,   0.0),
    (-90.0, -90.0, 0.0),
    (0.0, 0.0, 0.0),
]

# --- Generar trayectoria suave con muchos puntos intermedios ---
trayectoria_suave = []
pasos = 30  # más grande = movimiento más suave

for (q1a, q2a, d3a), (q1b, q2b, d3b) in zip(trayectoria[:-1], trayectoria[1:]):
    for alpha in np.linspace(0, 1, pasos, endpoint=False):
        q1 = (1 - alpha) * q1a + alpha * q1b
        q2 = (1 - alpha) * q2a + alpha * q2b
        d3 = (1 - alpha) * d3a + alpha * d3b
        trayectoria_suave.append((q1, q2, d3))

# agregamos la última posición exacta
trayectoria_suave.append(trayectoria[-1])

# --- Crear carpeta y frames ---
os.makedirs("media", exist_ok=True)
frames = []

for i, (q1_deg, q2_deg, d3_mm) in enumerate(trayectoria_suave):
    x_mm, y_mm, z_mm = fk_scara(q1_deg, q2_deg, d3_mm)

    q1_rad = np.deg2rad(q1_deg)
    q2_rad = np.deg2rad(q2_deg)

    x0, y0 = 0.0, 0.0
    x1 = L1 * np.cos(q1_rad)
    y1 = L1 * np.sin(q1_rad)
    x2 = x1 + L2 * np.cos(q1_rad + q2_rad)
    y2 = y1 + L2 * np.sin(q1_rad + q2_rad)

    fig, ax = plt.subplots()
    ax.plot([x0, x1, x2], [y0, y1, y2], marker="o")
    ax.set_xlabel("X [mm]")
    ax.set_ylabel("Y [mm]")
    ax.set_title(f"Frame {i+1} - q1={q1_deg:.1f}°, q2={q2_deg:.1f}°, d3={d3_mm:.1f} mm")
    ax.set_xlim(-200, 200)
    ax.set_ylim(-200, 200)
    ax.set_aspect("equal", "box")
    ax.grid(True)

    filename = f"media/frame_{i:03d}.png"
    fig.savefig(filename)
    plt.close(fig)

    frames.append(imageio.imread(filename))

# --- Guardar GIF: duration = segundos por frame (ajusta si quieres más lento/rápido) ---
gif_path = "media/scara_robot_fisico.gif"
imageio.mimsave(gif_path, frames, duration=0.12)  # 0.08 s por frame ≈ 12.5 fps

print("GIF generado en:", gif_path)


GIF generado en: media/scara_robot_fisico.gif
